# Assignment 5 — KNN Classification of Student Performance (Pass/Fail)
**Dataset:** UCI Student Performance (`student-por.csv`)  
**Target:** `Pass = 1 if G3 >= 10 else 0` (G1/G2/G3 excluded from features)

**Aim:** Standardize/encode features, scan K, select best-K by cross-validation, and evaluate with train/test curves, confusion matrix, report and ROC-AUC.


## 1. Library imports

In [1]:
import io, os
import urllib.request, zipfile
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
    roc_curve, roc_auc_score)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs("outputs", exist_ok=True)
print("random_state =", RANDOM_STATE)

random_state = 42


## 2. Dataset loading (official UCI source)

In [2]:
UCI_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"
raw = urllib.request.urlopen(UCI_URL, timeout=60).read()
zf = zipfile.ZipFile(io.BytesIO(raw))
with zf.open("student-por.csv") as f:
    df = pd.read_csv(f, sep=";")
print("Shape:", df.shape)

Shape: (649, 33)


## 3. Dataset preview

In [3]:
df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [4]:
df["Pass"] = (df["G3"] >= 10).astype(int)
print(df["Pass"].value_counts())
print("Pass rate:", df["Pass"].mean().round(4))

Pass
1    549
0    100
Name: count, dtype: int64
Pass rate: 0.8459


## 4. Data preprocessing — standardized + encoded features

In [5]:
X_raw = df.drop(columns=["G1", "G2", "G3", "Pass"])
y = df["Pass"].to_numpy()
cat_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_raw.select_dtypes(exclude=["object"]).columns.tolist()
print(f"Features: {X_raw.shape[1]}; numeric={len(num_cols)}, categorical={len(cat_cols)}")
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(f"Train {X_train_raw.shape} Test {X_test_raw.shape}")

Features: 30; numeric=13, categorical=17
Train (519, 30) Test (130, 30)


C:\Users\ns479\AppData\Local\Temp\ipykernel_27352\2144761865.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_raw.select_dtypes(include=["object"]).columns.tolist()


## 5. Model implementation — accuracy for different K + CV best-K

In [6]:
k_range = list(range(1, 31, 2))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_means, train_accs, test_accs = [], [], []
for k in k_range:
    pipe = Pipeline([("prep", preprocess), ("knn", KNeighborsClassifier(n_neighbors=k))])
    cv_means.append(cross_val_score(pipe, X_train_raw, y_train, cv=cv, scoring="accuracy").mean())
    pipe.fit(X_train_raw, y_train)
    train_accs.append(accuracy_score(y_train, pipe.predict(X_train_raw)))
    test_accs.append(accuracy_score(y_test, pipe.predict(X_test_raw)))
    print(f"k={k:2d} cv_acc={cv_means[-1]:.4f} train={train_accs[-1]:.4f} test={test_accs[-1]:.4f}")
best_k = k_range[int(np.argmax(cv_means))]
print(f"\nBest K by 5-fold CV accuracy: {best_k} (cv={max(cv_means):.4f})")
pd.DataFrame({"k": k_range, "cv_accuracy": cv_means, "train_accuracy": train_accs, "test_accuracy": test_accs}).round(4).to_csv("outputs/k_scan.csv", index=False)

k= 1 cv_acc=0.8478 train=1.0000 test=0.7923


k= 3 cv_acc=0.8516 train=0.9114 test=0.8154


k= 5 cv_acc=0.8497 train=0.8786 test=0.8077


k= 7 cv_acc=0.8516 train=0.8651 test=0.8154


k= 9 cv_acc=0.8555 train=0.8632 test=0.8154


...

k=21 cv_acc=0.8535 train=0.8651 test=0.8231


k=23 cv_acc=0.8497 train=0.8613 test=0.8231


k=25 cv_acc=0.8458 train=0.8574 test=0.8231


k=27 cv_acc=0.8458 train=0.8574 test=0.8231


k=29 cv_acc=0.8477 train=0.8536 test=0.8231

Best K by 5-fold CV accuracy: 13 (cv=0.8574)


In [7]:
plt.figure(figsize=(9, 5.5))
plt.plot(k_range, train_accs, marker="o", label="Train accuracy")
plt.plot(k_range, test_accs, marker="s", label="Test accuracy")
plt.plot(k_range, cv_means, marker="^", linestyle="--", label="5-fold CV accuracy (train)")
plt.axvline(best_k, color="red", linestyle=":", label=f"Best K={best_k}")
plt.xlabel("K (n_neighbors)"); plt.ylabel("Accuracy"); plt.title("KNN: accuracy vs K")
plt.legend(); plt.tight_layout(); plt.savefig("outputs/accuracy_vs_k.png", dpi=150); plt.close()
print("Saved: outputs/accuracy_vs_k.png")

Saved: outputs/accuracy_vs_k.png


## 6. Evaluation — final model (best K): report, confusion matrix, ROC-AUC

In [8]:
final = Pipeline([("prep", preprocess), ("knn", KNeighborsClassifier(n_neighbors=best_k))])
final.fit(X_train_raw, y_train)
y_pred = final.predict(X_test_raw)
y_proba = final.predict_proba(X_test_raw)[:, 1]
print(f"Best K={best_k} | train_acc={accuracy_score(y_train, final.predict(X_train_raw)):.4f} | test_acc={accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print("\nClassification report:"); print(classification_report(y_test, y_pred, target_names=["Fail(0)", "Pass(1)"]))
cm = confusion_matrix(y_test, y_pred); print("Confusion matrix:"); print(cm)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Pred Fail", "Pred Pass"], yticklabels=["True Fail", "True Pass"])
axes[0].set_title(f"Confusion matrix (K={best_k})")
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, label=f"KNN K={best_k} (AUC={roc_auc_score(y_test, y_proba):.3f})")
axes[1].plot([0, 1], [0, 1], "k--", label="Chance")
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR"); axes[1].set_title("ROC curve"); axes[1].legend()
plt.tight_layout(); plt.savefig("outputs/confusion_roc.png", dpi=150); plt.close()
print("Saved: outputs/confusion_roc.png")

Best K=13 | train_acc=0.8632 | test_acc=0.8231
ROC-AUC: 0.6639

Classification report:
              precision    recall  f1-score   support

     Fail(0)       0.20      0.05      0.08        20
     Pass(1)       0.85      0.96      0.90       110

    accuracy                           0.82       130
   macro avg       0.52      0.51      0.49       130
weighted avg       0.75      0.82      0.78       130

Confusion matrix:
[[  1  19]
 [  4 106]]


Saved: outputs/confusion_roc.png


## 7. Result and conclusion
- Features standardized/encoded in a pipeline; K scanned over odd values 1–29; best-K chosen by 5-fold stratified CV (`random_state=42`).
- Train/test accuracy graph, per-K table, confusion matrix, report and ROC-AUC saved to `outputs/`; all values computed at runtime.
